# Scientific Programming with Python 
## (Winter 2025/26)

# Session 06: Pandas

* Some more python: functions
* Pandas

# Functions in Python

## What is a function?

- A named block of code that takes inputs (parameters) and returns a result.
- Encapsulates logic for reuse, testing, and clarity.
- In Python, functions are first-class objects (you can store, pass, and return them).

In [ ]:
def add(a, b):
    return a + b

result = add(2, 3)
print(result)  # 5

### Defining and calling functions

- `def name(params): ... return value`
- A function without an explicit `return` returns `None`.

In [ ]:
def greet(name):
    print(f"Hello, {name}!")

greet("Ada")      # side-effect (prints)
x = greet("Bob")  # x is None

### Parameters vs. arguments; positional and keyword

- Parameters: names in the function definition; arguments: values at call-site.
- Positional vs. keyword arguments improve clarity and flexibility.

In [ ]:
def power(base, exp=2):
    return base ** exp

print(power(3))              # 9 (positional)
print(power(base=2, exp=5))  # 32 (keyword)

## Default arguments and the mutable default pitfall

- Default expressions are evaluated once at function definition time.
- Never use a mutable object (e.g., list) as a default unless you want reuse.

In [ ]:
def append_bad(x, acc=[]):        # BAD: acc persists across calls
    acc.append(x)
    return acc

print(append_bad(1))   # [1]
print(append_bad(2))   # [1, 2]  (surprise)

In [ ]:
def append_good(x, acc=None):     # GOOD: use sentinel, then create
    if acc is None:
        acc = []
    acc.append(x)
    return acc

print(append_good(1))  # [1]
print(append_good(2))  # [2]

### Variable positional and keyword arguments

- Variable positional arguments collect extra positional args into a tuple named with a star: `*args`.
- Variable keyword arguments collect extra named args into a dict: `**kwargs`.
- Useful for flexible APIs and for forwarding arguments to other functions.
- Parameter order (simplified): positional-only (optional) /, regular params, defaults, *args, keyword-only params (after a bare *), **kwargs.

In [ ]:
def summarize(title, *values, scale=1.0, **opts):
    """Sum arbitrary values with an optional scale; supports extra options via **opts."""
    total = sum(values) * scale
    if opts.get("verbose", False):
        print(f"{title}: values={values}, scale={scale}")
    return total

print(summarize("sum", 1, 2, 3, 4))                      # 10
print(summarize("scaled sum", 1, 2, 3, scale=0.5))       # 3.0
print(summarize("report", 2, 4, 6, scale=2, verbose=True))

Call-site unpacking with * and ** spreads iterables and dicts into arguments.

In [ ]:
nums = [10, 20, 30]
opts = {"scale": 0.1, "verbose": True}
print(summarize("unpacked", *nums, **opts))  # 6.0 with verbose print

- Library functions that accept variable args:
  - `print(*objects, sep=" ", end="\n")`
  - `max(a, b, c, ...)` and `min(...)`
  - `logging.info(fmt, *args)` (in logging module), `str.format`, etc.

### Scope and the LEGB rule

- Name resolution: Local, Enclosing, Global, Built-in.
- Closures capture variables from enclosing scopes.

In [ ]:
def make_adder(delta):
    def add(x):
        return x + delta   # delta from enclosing scope
    return add

inc5 = make_adder(5)
print(inc5(10))  # 15

### Modifying enclosed state with nonlocal

- Use `nonlocal` to rebind a variable from an enclosing (non-global) scope.

In [ ]:
def counter():
    n = 0
    def next_():
        nonlocal n
        n += 1
        return n
    return next_

c = counter()
print(c(), c(), c())  # 1 2 3

### First-class functions

- Functions can be assigned to variables, stored in containers, and passed around.

In [ ]:
def square(x): return x*x
def cube(x): return x*x*x

funcs = [square, cube]
for f in funcs:
    print(f.__name__, f(3))  # square 9, cube 27

### Anonymous functions (lambda)

- Short, single-expression functions; good for small “throwaway” callbacks.
- Limitations: expression-only (no statements), readability matters.

In [ ]:
# As key function
data = ["b", "aaa", "cc"]
print(sorted(data, key=lambda s: len(s)))  # by length

In [ ]:
# As a quick transform
inc = lambda x: x + 1
print(inc(10))  # 11

In [ ]:
ops = {"add": lambda a,b: a+b, "mul": lambda a,b: a*b}
print(ops["mul"](3, 4))  # 12

### Lambdas and closures; loop capture pitfall

- Lambdas capture variables by reference; common pitfall in loops.

In [ ]:
funcs = [lambda: i for i in range(3)]
print([f() for f in funcs])  # [2, 2, 2] (all refer to final i)

Fix with default arg binding

In [ ]:
funcs_fixed = [lambda i=i: i for i in range(3)]
print([f() for f in funcs_fixed])  # [0, 1, 2]

### Passing functions to your own functions (higher-order)

- Write utilities that accept a function argument to customize behavior.

In [ ]:
def apply_n_times(f, x, n):
    """Apply f to x, n times."""
    for _ in range(n):
        x = f(x)
    return x

print(apply_n_times(lambda z: 2*z, 3, 4))  # 48

### Standard library: sorted with key= (takes a function)

- `sorted(iterable, key=func)` calls func on each element to determine ordering.
- Also works with `min`/`max` and `list.sort`.

In [ ]:
words = ["Banana", "apple", "cherry", "Apricot"]
print(sorted(words, key=str.lower))
print(sorted(words, key=lambda s: (len(s), s.lower())))

### map, filter, any, all

- `map(func, iterable)` transforms items; `filter(func, iterable)` keeps those where func returns `True`.
- `any`/`all` consume iterables of booleans.

In [ ]:
nums = [1, 2, 3, 4, 5]
squares = list(map(lambda x: x*x, nums))
evens = list(filter(lambda x: x % 2 == 0, nums))
print(squares, evens)  # [1,4,9,16,25] [2,4]
print(any(x > 4 for x in nums), all(x > 0 for x in nums))  # True True

### functools: reduce, partial, cmp_to_key
- `reduce(func, iterable)` aggregates; `partial` pre-fills function arguments.

In [ ]:
from functools import reduce, partial

prod = reduce(lambda a,b: a*b, [1,2,3,4], 1)  # ((((1*1)*2)*3)*4)
print(prod)  # 24

In [ ]:
def power(base, exp): return base ** exp
square = partial(power, exp=2)
print(square(7))  # 49

### Writing a function that takes a function: numerical derivative

Example: central-difference derivative builder.

In [ ]:
def derivative(f, h = 1e-6):
    return lambda x: (f(x + h) - f(x - h)) / (2*h)

import math
d_sin = derivative(math.sin)
print(d_sin(0.0), "≈", math.cos(0.0))  # ~1.0 ≈ 1.0

Also works with numpy:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
x = np.linspace(0,2*np.pi,100)
plt.figure()
plt.plot(x, np.sin(x), label="sin")
plt.plot(x, derivative(np.sin)(x), label="d_sin")
plt.legend()
plt.show()

### Writing a function that takes a function: numerical integration

- Trapezoidal rule integrator that accepts f.

In [ ]:
from typing import Callable

def integrate_trap(f, a, b, n = 1000) -> float:
    h = (b - a) / n
    x = a
    total = 0.5 * (f(a) + f(b))
    for k in range(1, n):
        x = a + k*h
        total += f(x)
    return total * h

import math
approx = integrate_trap(math.sin, 0.0, math.pi, n=10000)
print(round(approx, 6))  # ~2.0

### Callbacks: timing or retry wrappers

- Higher-order utility that accepts a function to run.

In [ ]:
import time

def time_it(f, *args, **kwargs):
    t0 = time.perf_counter()
    out = f(*args, **kwargs)
    dt = time.perf_counter() - t0
    return out, dt

def work(n): return sum(range(n))
res, elapsed = time_it(work, 10_000)
print(res, f"{elapsed*1e3:.2f} ms")

Example: Generators and higher-order iteration

- Generator functions yield values lazily; can take function predicates.

In [ ]:
def take_until(iterable, pred):
    for x in iterable:
        if pred(x):
            break
        yield x

print(list(take_until(range(10), lambda k: k > 5)))  # [0,1,2,3,4,5]

Example: putting it together: customize behavior via a strategy function

In [ ]:
def transform_each(data, transform, pred=lambda _: True):
    """Apply transform(x) to each x satisfying pred(x)."""
    out = []
    for x in data:
        out.append(transform(x) if pred(x) else x)
    return out

nums = [1, 2, 3, 4, 5]
doubled_evens = transform_each(nums, lambda x: 2*x, pred=lambda x: x % 2 == 0)
print(doubled_evens)  # [1, 4, 3, 8, 5]

### Best practices and summary

- Keep functions small, single-purpose; name them clearly.
- Prefer explicit keyword arguments for clarity.
- Don't overuse lambda; define def for reuse or readability.
- Avoid mutable default arguments; use `None` sentinel.
- Test functions in isolation; document with docstrings.
- Functions are first-class: pass them to library tools (sorted, map), to your own utilities, and return them for customization.

# Scientific Programming with Python 
## (Winter 2024/25)

# Session 06: Pandas -- Series and DataFrames

# Pandas -- Series and DataFrames

Main source: https://github.com/jakevdp/PythonDataScienceHandbook/tree/master/notebooks

#### Pandas is a library for fast and efficient computation on big datasets. As in Numpy, many operations in Pandas are vectorized and thus efficient and fast.


Pandas is a newer package built on top of NumPy, and provides an efficient implementation of a DataFrame. DataFrames are essentially multidimensional arrays with attached row and column labels, and often with heterogeneous types and/or missing data. As well as offering a convenient storage interface for labeled data, Pandas implements a number of powerful data operations familiar to users of both database frameworks (-> relational algebra) and spreadsheet programs.

As we saw, NumPy's ndarray data structure provides essential features for the type of clean, well-organized data typically seen in numerical computing tasks. While it serves this purpose very well, its limitations become clear when we need more flexibility (e.g., attaching labels to data, working with missing data, etc.) and when attempting operations that do not map well to element-wise broadcasting (e.g., groupings, pivots, etc.), each of which is an important piece of analyzing the less structured data available in many forms in the world around us. Pandas, and in particular its Series and DataFrame objects, builds on the NumPy array structure and provides efficient access to these sorts of "data munging" tasks that occupy much of a data scientist's time.

In [ ]:
# Just as we import numpy usually as np, we import pandas under the alias of pd. 
# We'll import numpy as well, because we'll need it often when using pandas
import numpy as np
import pandas as pd

## The Pandas Series Object
A Pandas Series is a one-dimensional array of indexed data. It can be created from a list or array as follows:

In [ ]:
# missing values 
data = pd.Series([0.25, 0.5, np.NaN, 1.0])
data

In [ ]:
type(data)

In [ ]:
data.values, type(data.values)

In [ ]:
#The index is an array-like object of type pd.Index
data.index, type(data.index), list(data.index)

Like with a NumPy array, data can be accessed by the associated index via the familiar Python square-bracket notation:

In [ ]:
data[1:3]

In [ ]:
type(data[1])

In [ ]:
print(dir(data))

### Series as generalized NumPy array

From what we've seen so far, it may look like the Series object is basically interchangeable with a one-dimensional NumPy array. The essential difference is the presence of the index: while the Numpy Array has an implicitly defined integer index used to access the values, the Pandas Series has an explicitly defined index associated with the values.

This explicit index definition gives the Series object additional capabilities. For example, the index need not be an integer, but can consist of values of any desired type. For example, if we wish, we can use strings as an index:

In [ ]:
data = pd.Series([0.25, 0.5, 0.75, 1.0],
                 index=['a', 'b', 'd', 'c'])
data

In [ ]:
data.index = list("AbCD")
data

In [ ]:
data['b']

The use of an implicit index is now discouraged when an explicit index is present:

In [ ]:
data[1]

In [ ]:
data.iloc[1]

In [ ]:
data.values[1]

In [ ]:
data = pd.Series([0.25, 0.5, 0.75, 1.0],
                 index=[3, 7, 2, 4])
data

In [ ]:
data.index

When an explicit index is present, it is preferred! (*as long as we don't slice!*)

In [ ]:
data[3]

In [ ]:
data[1:3]

Note that explicit indices don't need to be unique!

In [ ]:
d2 = pd.concat([data, data])
d2

In [ ]:
d2[7]

In [ ]:
d2[7] = 2

In [ ]:
d2

### Series as specialized dictionary

In this way, you can think of a Pandas Series a bit like a specialization of a Python dictionary. A dictionary is a structure that maps arbitrary keys to a set of arbitrary values, and a Series is a structure which maps typed keys to a set of typed values. This typing is important: just as the type-specific compiled code behind a NumPy array makes it more efficient than a Python list for certain operations, the type information of a Pandas Series makes it much more efficient than Python dictionaries for certain operations.

The Series-as-dictionary analogy can be made even more clear by constructing a Series object directly from a Python dictionary:

In [ ]:
population_dict = {'California': 38332521,
                   'Texas': 26448193,
                   'New York': 19651127,
                   'Florida': 19552860,
                   'Illinois': 12882135}
population = pd.Series(population_dict)
population

In [ ]:
population['Texas']

Unlike a dictionary, though, the Series also supports array-style operations such as slicing:

In [ ]:
population['California':'Illinois']
# note that Illinois is included!

### Constructing Series objects

In [ ]:
# data can be a scalar
pd.Series(5, index=[100, 200, 300])

In [ ]:
# data can be a dictionary, 
ser = pd.Series({2:'a', 1:'b', 3:'c'})
ser

In [ ]:
ser.to_dict()

In [ ]:
list(ser.values)

## The Pandas DataFrame Object

The next fundamental structure in Pandas is the DataFrame. Like the Series object, it can be thought of either as a generalization of a NumPy array, or as a specialization of a Python dictionary. We'll now take a look at each of these perspectives.

### DataFrame as a generalized NumPy array

If a Series is an analog of a one-dimensional array with flexible indices, a DataFrame is an analog of a two-dimensional array with both flexible row indices and flexible column names. Just as you might think of a two-dimensional array as an ordered sequence of aligned one-dimensional columns, you can think of a DataFrame as a sequence of aligned Series objects. Here, by "aligned" we mean that they share the same index.



To demonstrate this, let's first construct a new Series listing the area of each of the five states discussed in the previous section:

In [ ]:
area_dict = {'California': 423967, 'Texas': 695662, 'New York': 141297,
             'Florida': 170312, 'Illinois': 149995}
area = pd.Series(area_dict)
area

Now that we have this along with the population Series from before, we can use a dictionary to construct a single two-dimensional object containing this information:

In [ ]:
states = pd.DataFrame({'population': population,
                       'area': area,
                       'country': 'USA'})
states

In [ ]:
print(states.dtypes)

This looks like a generalized dictionary! The keys are the names of the state, and the values are like a list [area, country, population]

In [ ]:
states.sort_values(by="population", ascending=False)

In [ ]:
states['population'], type(states['population'])

In [ ]:
states["population"].idxmax() #figures out the "key(s)"(indices) of the DataFrame where "population" has its max

In [ ]:
states.loc[states["population"].idxmax()] #returnes the series at the given index

In [ ]:
states.max() #returns the max for every individual column!

In [ ]:
states['California']

In [ ]:
states.loc['California']

In [ ]:
states.index

In [ ]:
states.columns

In [ ]:
states.values

In [ ]:
type(states.values)

Thus the DataFrame can be thought of as a generalization of a two-dimensional NumPy array, where both the rows and columns have a generalized index for accessing the data.

## DataFrame as specialized dictionary

Similarly, we can also think of a DataFrame as a specialization of a dictionary. Where a dictionary maps a key to a value, a DataFrame maps a column name to a Series of column data. For example, asking for the 'area' attribute returns the Series object containing the areas we saw earlier:

In [ ]:
states["area"]
# note that indexing a DataFrame with square brackets gets the *column*!

In [ ]:
type(states["area"])

### Constructing DataFrame objects

A Pandas DataFrame can be constructed in a variety of ways:

#### From a single Series object

A DataFrame is a collection of Series objects, and a single-column DataFrame can be constructed from a single Series:

In [ ]:
population

In [ ]:
pd.DataFrame(population, columns=['population'])

#### From multiple Series

In [ ]:
s1 = pd.Series(['100', '200', 'python', '300.12', '400'])
s2 = pd.Series(['10', '20', 'php', '30.12', '40'])
df = pd.concat([s1, s2], axis='columns')
df

Note that many functions in pandas take the `axis` argument - In which case you can select between 0/`'index'` and  1/`'columns'`. If you want to be explicit, I recommend using the string-version!

#### From a list of dicts 

Any list of dictionaries can be made into a DataFrame. We'll use a simple list comprehension to create some data. Even if some keys in the dictionary are missing, Pandas will fill them in with NaN (i.e., "not a number") values:

In [ ]:
df = pd.DataFrame([{'a': 1, 'b': 2}, {'b': 3, 'c': 4}], index=["first_dict", "second_dict"])
df

As every single column must have a consistent dtype and np.NaN is a float, some of the numbers get coerced into floats:

In [ ]:
df['a']

In [ ]:
df['b']

In [ ]:
type(np.NaN)

In [ ]:
df.dtypes

If we wanted to get the rows, pandas would need to coerce the numbers explicitly: 

In [ ]:
df

In [ ]:
df.loc['first_dict']

#### From a two-dimensional NumPy array

Given a two-dimensional array of data, we can create a DataFrame with any specified column and index names. If omitted, an integer index will be used for each:


In [ ]:
dates = pd.date_range('20130101', periods=6)
df = pd.DataFrame(np.random.randn(6,4), index=dates, columns=list('ABCD'))
df

## The Pandas Index Object

We have seen here that both the Series and DataFrame objects contain an explicit index that lets you reference and modify data. This Index object is an interesting structure in itself, and it can be thought of as an immutable array:

In [ ]:
ind = pd.Index([2, 3, 5, 7, 11])
ind

In [ ]:
ind[0] = 1

In [ ]:
sr = pd.Series(0, index=ind)
sr

Index objects have a name:

In [ ]:
ind.names = ['indexx']
ind

In [ ]:
sr = pd.Series(np.zeros_like(ind), index=ind)
sr

In [ ]:
df = pd.DataFrame(np.zeros_like(ind), index=ind, columns=['first'])
df

In [ ]:
df.index.names = [None]
df

Index objects also have many of the attributes familiar from NumPy arrays:

In [ ]:
ind.size, ind.shape, ind.ndim, ind.dtype

# Data Indexing, Selection, and Assignment


From the numpy lecture, we already know about indexing, slicing, masking, and fancy indexing:

In [ ]:
a = np.arange(16).reshape(4,4)
a

In [ ]:
a[:, [1, 3]][a[:, [1, 3]] % 3 == 0]
# Takes those values of the second and fourth column that are divisible by 3

Here we'll look at similar means of accessing and modifying values in Pandas Series and DataFrame objects. The corresponding patterns in Pandas are very similar to those of numpy, though there are a few quirks to be aware of.

We'll start with the simple case of the one-dimensional Series object, and then move on to the slightly more complicated two-dimensional DataFrame object.

## Data Selection in Series

As we saw in the previous section, a Series object acts in many ways like a one-dimensional NumPy array, and in many ways like a standard Python dictionary. If we keep these two overlapping analogies in mind, it will help us to understand the patterns of data indexing and selection in these arrays.

### Series as dictionary

Like a dictionary, the Series object provides a mapping from a collection of keys to a collection of values, which means most of the corresponding functions work just as well for them:

In [ ]:
data = pd.Series([0.25, 0.5, 0.75, 1.0],
                 index=['a', 'b', 'c', 'd'])
data

In [ ]:
data.__contains__('b')

In [ ]:
'b' in data

In [ ]:
np.array_equal(data.keys(), data.index)

In [ ]:
data

In [ ]:
list(data.items())

In [ ]:
data['e'] = 1.25
data

### Series as one-dimensional array

Series builds on this dictionary-like interface and provides array-style item selection via the same basic mechanisms as NumPy arrays – that is, slices, masking, and fancy indexing. Examples of these are as follows:

In [ ]:
# slicing by explicit index
data['a':'c']

In [ ]:
# slicing by implicit integer index
data[0:2] 
# Note that when slicing with an explicit index (i.e., data['a':'c']), the final index is included in the slice, 
# while when slicing with an implicit index (i.e., data[0:2]), the final index is excluded from the slice.

In [ ]:
(data > 0.3) & (data < 0.8)

In [ ]:
# masking
data[(data > 0.3) & (data < 0.8)]

In [ ]:
# fancy indexing
data[['a', 'e']]

In [ ]:
data = pd.Series([0.25, 0.5, 0.75, 1.0],
                 index=[1, 2, 3, 4])
data

In [ ]:
data[1:3]

**If your Series has an explicit integer index, an indexing operation such as data[1] will use the explicit indices, while a slicing operation like data[1:3] will use the implicit Python-style index.**

In [ ]:
data = pd.Series(['a', 'b', 'c'], index=[1, 5, 3])
data

In [ ]:
# explicit index when indexing
data[1]

In [ ]:
# implicit index when slicing
data[1:3]

The **loc** attribute allows indexing and slicing that *always* references the explicit index:

In [ ]:
data.loc[1]

In [ ]:
data.loc[1:3]

Note that `loc` may or may not throw Index-Errors when slicing:

In [ ]:
data = pd.Series(['a', 'b', 'c'], index=[1, 5, 3])
data

In [ ]:
data.loc['a':'z']

In [ ]:
data.loc[3:10]

In [ ]:
data = pd.Series(['a', 'b', 'c'], index=[1, 3, 5])
data.loc[3:10]

In [ ]:
data.loc['a':'z']

The **iloc** attribute allows indexing and slicing that always references the implicit Python-style index:

In [ ]:
data.iloc[1]

In [ ]:
data.iloc[1:3]

Please, save yourself the pain and be always explicit about what you do -- always use ``.loc`` and ``.iloc``

### Addendum: Reindexing

In [ ]:
s = pd.Series([1, 2, 3])
s

In [ ]:
s.loc[[1, 2, 3]]

instead, you should go for [`reindex`][1], which conforms the Series/DF to new index with optional filling logic.

[1]: https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.reindex.html

In [ ]:
s.reindex([1, 2, 3])

## Data Selection in DataFrame

Recall that a DataFrame acts in many ways like a two-dimensional or structured array, and in other ways like a dictionary of Series structures sharing the same index. These analogies can be helpful to keep in mind as we explore data selection within this structure.

In [ ]:
area = pd.Series({'California': 423967, 'Texas': 695662,
                  'New York': 141297, 'Florida': 170312,
                  'Illinois': 149995})
pop = pd.Series({'California': 38332521, 'Texas': 26448193,
                 'New York': 19651127, 'Florida': 19552860,
                 'Illinois': 12882135})
data = pd.DataFrame({'area':area, 'pop':pop, 'Country':'USA'})
data

Note that if we index a DataFrame, we index the **column**!!

In [ ]:
# Dictionary-style indexing results in a Series....
print(type(data["area"]))
data["area"]

In [ ]:
# We can also dereference, though it leads to side-effects if that's actually also a method...
data.area

In [ ]:
type(data.values)

With this picture in mind, many familiar array-like observations can be done on the DataFrame itself. For example, we can transpose the full DataFrame to swap rows and columns:

In [ ]:
data.T

For array-style indexing, Pandas again uses the loc and iloc indexers mentioned earlier. Using the iloc indexer, we can index the underlying array as if it is a simple NumPy array (using the implicit Python-style index), **but the DataFrame index and column labels are maintained in the result**:

In [ ]:
#indexing the underlying numpy-array...
data.values[:3, :2]

In [ ]:
data.iloc[:3, :2]

In [ ]:
data

In [ ]:
data.loc[:'Illinois', :'pop']

In [ ]:
data.loc[:,['area','pop']]

So, this is how we get a row!

In [ ]:
data.loc["California", :]

In [ ]:
# adding a new column.. (vectorized calculations!)
data['density'] = data['pop'] / data['area']
# we can combine masking with fancy indexing
data.loc[data.density > 100, ['pop', 'density']]

If you want to combine explicit and implicit indexing, you have to chain them:

In [ ]:
data.iloc[1:4].loc[:, ['pop', 'density']]

**While indexing refers to columns, slicing refers to rows:**

In [ ]:
data['area']

In [ ]:
data['Florida':'Illinois']

Again, rather be explicit about your indexing to save yourself from a lot of confusion.

In [ ]:
data['area':'pop']

In [ ]:
data.loc[:, 'area':'pop']

Fast access to a single member using **at**

In [ ]:
%%timeit
data.loc['Florida', 'pop']

In [ ]:
%%timeit
data.at['Florida', 'pop']

In [ ]:
data

|                        |              |                                          |                          |
|------------------------|--------------|------------------------------------------|--------------------------|
| **direct []-access**   |              |                                          |                          |
| One argument,   single | Column       | data['area']                             |                          |
| One argument,   slice  | Row          | data['Florida': 'Illinois']              | slice-top is included    |
| Both arguments         | only MultiInd| -                                        |                          |
| **.loc[]**             |              |                                          |                          |
| One argument,   single | Row          | data.loc['Florida']                      |                          |
| One argument,   slice  | Row          | data.loc['Florida': 'Illinois']          | slice-top  is included   |
| Both arguments, both   | Row, Column  | data.loc['Florida': 'Illinois', 'area']  | slice-top  is included   |
| **.iloc[]**            |              |                                          |                          |
| One argument,   single | Row          | data.iloc[0]                             |                          |
| One argument,   slice  | Row          | data.iloc[0, 2]                          | slice-top  is excluded   |
| Both arguments, both   | Row, Column  | data.iloc[0: 2, 0:3]                     | slice-top  is excluded   |

### Reindexing

In [ ]:
index = ['Firefox', 'Chrome', 'Safari', 'IE10', 'Konqueror']
df = pd.DataFrame({'http_status': [200, 200, 404, 404, 301],
                  'response_time': [0.04, 0.02, 0.07, 0.08, 1.0]},
                  index=index)
df

In [ ]:
new_index = ['Safari', 'Iceweasel', 'Comodo Dragon', 'IE10', 'Chrome']
df.reindex(new_index)

In [ ]:
df.reindex(columns=['http_status', 'user_agent'])

**Renaming indices** 

In [ ]:
df = pd.DataFrame({"a": [1, 2, 3, 4], "b": [2, 5, 7, 8]})
df = df.rename(mapper={'b': 'c'}, axis='columns')
df

### Boolean Indexing

In [ ]:
dates = pd.date_range('20130101', periods=6)
df = pd.DataFrame(np.random.randn(6,4), index=dates, columns=list('ABCD'))
df['E'] = ["one", "two", "three"] * 2
df

In [ ]:
df['A'] > 0.5

In [ ]:
df[df['A'] > 0]

In [ ]:
#alternate syntax: `query` - see https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.query.html
df.query('A > 0')

In [ ]:
df['E'].isin(['one','two'])

In [ ]:
df[df['E'].isin(['one','two'])] = np.NaN
df

In [ ]:
pd.isna(df)

In [ ]:
pd.isna(df).any(axis=1)

In [ ]:
df[~df.isna().any(axis=1)]

In [ ]:
df.dropna(how="any")

## Data assignment

In [ ]:
df = pd.DataFrame({'temp_c': [17.0, 25.0]},
                  index=['Portland', 'Berkeley'])
df

### Assigning columns

In [ ]:
df['country'] = 'USA'
df

In [ ]:
df['temp_c'] <= 18

In [ ]:
df['too_cold'] = df['temp_c'] <= 18
df

These however work in-place. To assign to a new dataframe, use `assign`:

In [ ]:
df = pd.DataFrame({'temp_c': [17.0, 25.0]},
                  index=['Portland', 'Berkeley'])
df

In [ ]:
df2 = df.assign(temp_f=lambda x: x.temp_c * 9 / 5 + 32)
df2

In [ ]:
#vectorized version:
df.assign(temp_f=df['temp_c'] * 9 / 5 + 32)

In [ ]:
#multiple assignments simulatenously:
df.assign(temp_f=lambda x: x['temp_c'] * 9 / 5 + 32,
          temp_k=lambda x: (x['temp_f'] +  459.67) * 5 / 9)

In [ ]:
df

### Row assignment

In [ ]:
df.loc['Berkeley', 'temp_c'] = 26.0
df

In [ ]:
type(df.loc['Portland'])

In [ ]:
df.loc['Portland'] = pd.Series({'temp_c': 99})
df

In [ ]:
df.loc['Osnabrück', 'temp_c'] = 18
df

In [ ]:
df = pd.concat([df, df])
df

In [ ]:
df.loc['Osnabrück', 'temp_c'] = 25
df

In [ ]:
df.loc['Osnabrück'] = pd.Series({'temp_c': 99})
df

In [ ]:
type(df.loc['Osnabrück'])

In [ ]:
np.where(df.index == 'Osnabrück')

In [ ]:
df.iloc[np.where(df.index == 'Osnabrück')[0][0]]

In [ ]:
df.iloc[np.where(df.index == 'Osnabrück')[0][0]] = pd.Series({'temp_c': 99})
df

## Pandas indexing

### Multi-Indexing

While Pandas does provide objects that natively handle three-dimensional and four-dimensional data, a far more common pattern in practice is to make use of `hierarchical indexing` (also known as `multi-indexing`) to incorporate multiple index levels within a single index. In this way, higher-dimensional data can be compactly represented within the familiar one-dimensional Series and two-dimensional DataFrame objects.

In [ ]:
index = [('California', 2000), ('California', 2010),
         ('New York', 2000), ('New York', 2010),
         ('Texas', 2000), ('Texas', 2010)]
populations = [33871648, 37253956,
               18976457, 19378102,
               20851820, 25145561]
pop = pd.Series(populations, index=index)
pop

In [ ]:
index = pd.MultiIndex.from_tuples(index)
index

In [ ]:
index.names = ['state', 'year']

In [ ]:
pop = pop.reindex(index)
pop

In [ ]:
pop['California', 2000], pop['California', 2010]

In [ ]:
pop.iloc[0]

In [ ]:
pop.iloc[1]

### MultiIndex as extra dimension: stack() and unstack()

You might notice something else here: we could easily have stored the same data using a simple ``DataFrame`` with index and column labels.
In fact, Pandas is built with this equivalence in mind. The ``unstack()`` method will quickly convert a multiply indexed ``Series`` into a conventionally indexed ``DataFrame``:

In [ ]:
pop.unstack()

In [ ]:
index.names = [None, None]
pop = pop.reindex(index)
pop

In [ ]:
pop.unstack()

In [ ]:
pop.index.names = [None, None]
pop.unstack().T

In [ ]:
popdf = pop.unstack(level=0)
popdf

In [ ]:
popdf.stack()

### Index setting and resetting

Another way to rearrange hierarchical data is to turn the index labels into columns; this can be accomplished with the ``reset_index`` method.
Calling this on the population dictionary will result in a ``DataFrame`` with a *state* and *year* column holding the information that was formerly in the index.
For clarity, we can optionally specify the name of the data for the column representation:

In [ ]:
pop

In [ ]:
pop.index.names = ['state', 'year']
print(type(pop))
pop

In [ ]:
pop_flat = pop.reset_index(name='population')
print(type(pop_flat))
pop_flat

Often when working with data in the real world, the raw input data looks like this and it's useful to build a ``MultiIndex`` from the column values.
This can be done with the ``set_index`` method of the ``DataFrame``, which returns a multiply indexed ``DataFrame``:

In [ ]:
pop_df = pop_flat.set_index(['state', 'year'])
pop_df

In [ ]:
pop_df.rename_axis([None, None])

In [ ]:
asdf = pop_df.rename_axis([None, None]).unstack()
asdf

In [ ]:
asdf.columns

In [ ]:
asdf["area"] = 999
asdf

In [ ]:
asdf.columns

In [ ]:
print(type(asdf["area"]))
asdf["area"]

In [ ]:
print(type(asdf["population"]))
asdf["population"]

In [ ]:
pop_flat

In [ ]:
pop_df2 = pop_flat.set_index('state').rename_axis(None)
pop_df2

In [ ]:
pop_df

In [ ]:
pop_df.reset_index()

# Reading Series and DataFrames

In [ ]:
%%bash
head Pokemon.csv

In [ ]:
df = pd.read_csv("Pokemon.csv")

Imagine someboy gave you a random dataset. You don't know any of its contents. What are the first steps you do?

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
df["Type 1"].value_counts()

In [ ]:
df["Legendary"].value_counts()

In [ ]:
#when working with csvs you really want to make sure if you want the first column as index-column!
df = pd.read_csv("Pokemon.csv", index_col=0)
df.tail()

In [ ]:
df.reset_index().tail()

In [ ]:
df.reset_index().drop_duplicates(subset="#").tail()

In [ ]:
df = df[df['Name'] != 'Volcanion']
df.tail()

In [ ]:
no_duplicates = df.reset_index().drop_duplicates(subset="#").reset_index().drop("index", axis=1)  
no_duplicates.tail()

In [ ]:
no_duplicates.set_index("#").to_csv('Pokemon_no_duplicates.csv')
#no_duplicates.to_excel('Pokemon_no_duplicates.xlsx', sheet_name='Sheet1')

In [ ]:
%%bash
head Pokemon_no_duplicates.csv

In [ ]:
gen_one = no_duplicates[no_duplicates["Generation"] == 1].set_index("#")
gen_one.tail()

In [ ]:
first_gen_dict = gen_one["Name"].to_dict()

In [ ]:
[str(key)+" : "+str(val) for index, (key, val) in enumerate(first_gen_dict.items()) if index < 9]

**Documentation!**

There are really really many arguments for this function, suiting all of your needs!

https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_csv.html

# Ufuncs and Aggregation

## Aggregation in Pandas

Aggregations are functions, where one or more dimensions of data are collapsed onto a single value, like the `max`, `sum` or `mean`- functions.

Stat-operations generally *exclude* missing data.

### For Series

In [ ]:
a = np.arange(7)
ser = pd.Series(a**2, index=a)
ser

In [ ]:
ser.sum()
#mean(), median(), min(), max(), ...

### For DataFrames

In [ ]:
df = pd.DataFrame({'A': a**2,
                   'B': a**3})
df

In [ ]:
df.mean()

In [ ]:
df.mean(axis=0)

In [ ]:
df.mean(axis='rows')

In [ ]:
df.mean(axis=1)

The following table summarizes some other built-in Pandas aggregations:

| Aggregation              | Description                     |
|--------------------------|---------------------------------|
| ``count()``              | Total number of items           |
| ``first()``, ``last()``  | First and last item             |
| ``mean()``, ``median()`` | Mean and median                 |
| ``min()``, ``max()``     | Minimum and maximum             |
| ``std()``, ``var()``     | Standard deviation and variance |
| ``mad()``                | Mean absolute deviation         |
| ``prod()``               | Product of all items            |
| ``sum()``                | Sum of all items                |

These are all methods of ``DataFrame`` and ``Series`` objects.

## Ufuncs


We know Ufuncs already from Numpy: It are vectorized functions that change all values of an array simultaneously. 

Pandas does the same, with a nice twist: for unary operations like negation and trigonometric functions, these ufuncs will *preserve index and column labels* in the output, and for binary operations such as addition and multiplication, Pandas will automatically *align indices* when passing the objects to the ufunc.


This means that keeping the context of data and combining data from different sources –both potentially error-prone tasks with raw NumPy arrays– become essentially foolproof ones with Pandas.

In [ ]:
rng = np.random.RandomState(0)
df = pd.DataFrame(rng.randint(0, 10, (3, 4)),
                  columns=['A', 'B', 'C', 'D'])
df

In [ ]:
np.exp(df)

### UFuncs: Index Alignment

For binary operations on two ``Series`` or ``DataFrame`` objects, Pandas will align indices in the process of performing the operation.
This is very convenient when working with incomplete data.

In [ ]:
area = pd.Series({'Alaska': 1723337, 'Texas': 695662,
                  'California': 423967}, name='area')
population = pd.Series({'California': 38332521, 'Texas': 26448193,
                        'New York': 19651127}, name='population')
area

In [ ]:
population

In [ ]:
area / population

In [ ]:
"divide" in dir(pd.DataFrame)

In [ ]:
popdens = area.divide(population, fill_value=0)
popdens

In [ ]:
popdens = popdens.replace([np.inf, -np.inf], np.nan)
popdens.dropna()

In [ ]:
A = pd.DataFrame(rng.randint(0, 20, (2, 2)),
                 columns=list('AB'))
A

In [ ]:
B = pd.DataFrame(rng.randint(0, 20, (3, 3)),
                 columns=list('ABC'))
B

In [ ]:
A+B

In [ ]:
A.add(B, fill_value=0)

### More Index-Alignment

In [ ]:
df = pd.DataFrame({'a': np.random.randint(3, size=10)}, index=np.arange(1, 20, 2))
df

Let's add a new column to this DataFrame!

In [ ]:
tmp = pd.Series([0]*len(df.index))
tmp

In [ ]:
#df['new'] = tmp   #changes the original one
df.assign(new=tmp) #creates a copy

In [ ]:
old_aligned, new_aligned = df.align(tmp, axis=0)
old_aligned

In [ ]:
new_aligned

In [ ]:
old_aligned.assign(new=new_aligned)

In [ ]:
tmp = pd.Series([0]*len(df.index), index=df.index)
tmp

In [ ]:
df['new'] = tmp
df

## .agg()

If you want to apply more than one operation (ufunc/aggregation), use `.agg()`:

In [ ]:
df = pd.DataFrame([[1, 2, 3],
                   [4, 5, 6],
                   [7, 8, 9],
                   [np.nan, np.nan, np.nan]],
                  columns=['A', 'B', 'C'])
df

In [ ]:
df.agg(['sum', 'min'])

In [ ]:
#you can also use different aggregations for different columns: 
df.agg({'A' : ['sum', 'min'], 'B' : ['min', 'max']})

## apply()

While some ufuncs (like cumsum or exp) are pre-defined by pandas, the method `apply` can be used to run an arbitrary function on all elements of a Series or DataFrame.

In [ ]:
a = np.arange(7)
df = pd.DataFrame({'A': a**2,
                   'B': a**3})
df

In [ ]:
df.cumsum()

In [ ]:
df.apply(np.cumsum)

In [ ]:
df["A_cumsum"] = df.cumsum()["A"]
df["B_cumsum"] = df.apply(np.cumsum)["B"]
df

Using Lambda-functions, we can combine `apply` with arbitrary functions. Note that the argument of the function is always an entire column of the dataset.

In [ ]:
df.apply(lambda x: print(x, end='\n\n'))

In [ ]:
df

In [ ]:
df['A'] + 1

In [ ]:
df.apply(lambda x: x+1)

In [ ]:
def my_more_complex_func(ser):
    res = []
    for elem in ser:
        print(elem if elem > 16 else -elem)
        res.append(elem if elem > 16 else -elem)
    return res

In [ ]:
df.apply(my_more_complex_func)

In [ ]:
df

In [ ]:
df.apply(lambda x: x.max() - x.min())

Note that `apply` works for both DataFrames and Series!

In [ ]:
df["A"].apply(lambda x: print(x))

In [ ]:
df["A_normed"] = df["A"].apply(lambda x: x/df["A"].max())
df

We can even use dictionaries with the apply-function!

In [ ]:
z_moves = {"Normal": "Breakneck Blitz", "Fighting": "All-Out Pummeling", "Flying": "Supersonic Skystrike", "Poison": "Acid Downpour", "Ground": "Tectonic Rage", "Rock": "Continental Crush", "Bug": "Savage Spin-Out", "Ghost": "Never-Ending Nightmare",
"Steel": "Corkscrew Crash", "Fire": "Inferno Overdrive", "Water": "Hydro Vortex", "Grass": "Bloom Doom", "Electric": "Gigavolt Havoc", "Psychic": "Shattered Psyche", "Ice": "Subzero Slammer", "Dragon": "Devastating Drake", "Dark": "Black Hole Eclipse", "Fairy": "Twinkle Tackle"}
df = pd.read_csv("Pokemon.csv")
df.head()

In [ ]:
df["Z-Move"] = df["Type 1"].apply(lambda x:z_moves[x])
df.head()

Using `apply`, we can also convert a list of Series into a DataFrame, by making the individual columns to Series:

In [ ]:
s = pd.Series([ ['Red', 'Green', 'White'], ['Red', 'Black'], ['Yellow']]) 
print(type(s))
s

In [ ]:
pd.Series([1, 2, 3])

In [ ]:
df = s.apply(pd.Series)
print(type(df))
df

**Note on speed:**

According to ([1]), `apply()` is twice as fast as looping through a df's `iterrows()`, and 8 times as fast as looping over python lists.

Note however, that while `apply()` is much faster at looping over the rows of your DataFrame/Series (by taking advantage of a number of internal optimizations, such as using iterators in Cython), it still inherently loops through rows. Whatever you're applying, you're still executing it once for every row. So, wherever you can use make use of vectorized Ufuncs, do so - that is far more optimized and parallelized - for ([1]) exchanging the haversine distance formula with it's vectorized counterpart led to a 50-fold-decrease in time!

\(1): https://engineering.upside.com/a-beginners-guide-to-optimizing-pandas-code-for-speed-c09ef2c6a4d6


[1]: https://engineering.upside.com/a-beginners-guide-to-optimizing-pandas-code-for-speed-c09ef2c6a4d6 

# Group-By

## Split-Apply-Combine

While simple operations are already pre-defined by pandas, custom aggregations and operations can be performed via **group-by**. The group-by operation can be described as having the following steps:

* **Splitting** the data into groups based on some criteria (breaking up and grouping depending on the value of a key)
* **Applying** a function to each group independently (aggregation, transformation, filtering, ...)
* **Combining** the results into a data structure

A typical example, for where the *apply* is a summerization aggregation, is illustrated here:

![](split-apply-combine.png)

In [ ]:
tmp = np.array([list("ABCABC"), np.arange(1,7)]).T
tmp

In [ ]:
df = pd.DataFrame(tmp, columns=["key", "data"])
df["data"] = pd.to_numeric(df["data"])
df

In [ ]:
df.groupby("key")

Note that what is returned is not a set of `DataFrames`, but a `DataFrameGroupBy` object. This object is where the magic is: you can think of it as a special view of the `DataFrames`, which is poised to dig into the groups but does no actual computation until the aggregation is applied. This "lazy evaluation" approach means that common aggregates can be implemented very efficiently in a way that is almost transparent to the user.

To produce a result, we can apply an aggregate to this `DataFrameGroupBy` object, which will perform the appropriate apply/combine steps to produce the desired result:

In [ ]:
df.groupby("key").sum().reset_index()

In [ ]:
df.groupby("key")["data"].sum()
# we can do column indexing just like on a normal DataFrame

### Iteration over groups

The ``GroupBy`` object supports direct iteration over the groups, returning each group as a ``Series`` or ``DataFrame``:

In [ ]:
df

In [ ]:
for (key, _) in df.groupby("key"):
    print(key)
    
print()
for (_, group) in df.groupby("key"):
    print(group, "\n")

In [ ]:
pkm = pd.read_csv('Pokemon.csv')
pkm.groupby('Generation')['Total'].mean()

### Dispatch methods

Any method not explicitly implemented by the ``GroupBy`` object will be passed through and called on the groups, whether they are ``DataFrame`` or ``Series`` objects.
For example, you can use the ``describe()`` method of ``DataFrame``s to perform a set of aggregations that describe each group in the data:

In [ ]:
df.describe()

In [ ]:
df.groupby("key").describe()

In [ ]:
df = pd.read_csv("Pokemon_no_duplicates.csv", index_col=0)
df.groupby('Generation')["Name"].nunique()

## References

Video tutorial from Pycon 2015

In [ ]:
from IPython.display import YouTubeVideo
YouTubeVideo('5JnMutdy6Fw')